# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

I re-read the two findings in the *FlyRank State of AI-Driven SEO, March 2026* paper that are closest to my Lane-2 work (refresh opportunity scoring). For each: where the label value comes from, and whether the validation design actually carries the claim.

### Finding #2 — "The Content Performance Curve" (health score by content age)

**The claim.** Content health peaks at 61–90 days (33.1), plateaus to ~180 days, hits a "decay cliff" at 271–365 days (14), then "recovers" at 365+ (25.1). The paper's safe reading is narrowed on the rebound: "older pages can recover **when they are updated well**; this is not evidence that age naturally reverses decline on its own."

**Where the label comes from.** The health score is a composite — impressions (30 pts) + position (30 pts) + CTR (20 pts) + scroll depth (20 pts) — and it is measured in the **current** snapshot, not over time. So this is a *cross-sectional* comparison: pages are grouped by how old they are *today*, and each page contributes one current health reading. It is not a single cohort followed from 30 to 365 days — it is 341,701 different pages frozen at whatever age they happen to be.

**Does the validation design carry the claim?** Partly, with two gaps I'd want closed before treating it as a lifecycle law:

1. **It is not an age-controlled outcome.** A page that is 365+ days old and still alive in the snapshot is a *survivor* of whatever made other 365+ pages disappear or lose all traffic. The paper itself admits this for the "Old + Long-Unchanged" cell in Finding #8: "the active-content subset introduces strong survivor bias there." The rebound column is exactly the cell where that bias bites. Health was also computed from an *active-content* subset (`impressions_90d > 0` and `sessions_90d > 0`), so zero-traffic old pages are filtered out before the curve is drawn.
2. **The "refresh" mechanism is asserted, not stratified.** The paper explains the 365+ recovery as "older pages that were refreshed" (Finding #4), but the age-curve table itself never splits refreshed vs not-refreshed — the control for the proposed mechanism lives in a different finding. The 57× impression and 3.2× health refresh figures come from a `365+, refreshed within 30 days` cell whose count I can't reconstruct from the public appendix, and Finding #8 already warns the peer cell is tiny.

**My takeaway for my own work.** The paper's honesty about these gaps (survivor bias, "not evidence that age reverses decline") is the model I want to copy. It also tells me: a cross-sectional "old pages look weak" table does not license me to claim that *refreshing* old pages will lift them. That is precisely the "declining ≠ refresh will pay off" trap my lane guide warns about.

### Finding #4 — "The Freshness Multiplier" (growth-to-decline ratio by freshness window)

**The claim.** The 31–90 day freshness band is the strongest stable growth window at 7.88:1 growth-to-decline. The `361+` bucket (283:1) is "visible but too small and too unstable to treat as a headline multiplier."

**Where the label comes from.** Each page is binned by days since last update, and the ratio is **growing-pages ÷ declining-pages** within that bin. "Growing" and "declining" come from 30-day-vs-previous-30-day impression change (the paper's own Trend Direction: Up > +10%, Down > −10%) — a label computed *within the same 30-day trend window*, not from a future outcome after a decision.

**Does the validation design carry the claim?** No — the paper knowingly exposes the failing case, which is exactly why it's the most instructive finding for me. The `361+` bucket is **283 growing pages against 1 declining page**. That single-declining-page cell means the 283:1 ratio has an error bar the width of a parking lot: if two more pages had flagged down, the ratio collapses to ~94:1; if the one decline were re-counted, it becomes undefined. The paper's own language — "the 361+ bar is present because it exists in the local sample" — concedes the number is not a measurement of a pattern but an artifact of near-empty cells.

**My takeaway.** Three habits to steal:

1. Report **cell counts, not just ratios** — a ratio without its denominator is not inspectable. (This is the same rule as reporting the base rate next to P@50.)
2. Demote **tiny cells** to "needs more data" before calling them findings — the paper demotes 361+, and I should demote thin `content_age` buckets in my own queues.
3. Same-window trend labels ("growing"/"declining" from a 30-day change) describe **current state, not future behavior** — a direction label is not a prediction of what happens next. My W5 label has exactly this weakness, and Section 2 confronts it.


## 2. My model under an honest split (before/after)

My Week-5 model already used a **client-holdout** split — the right *kind* of validation,
because pages from the same client share SEO patterns and a model can memorize the client
instead of learning content. But W5 reported **one single draw**: it held out just 20% of
clients (~6 of 32) with one random seed. Clients differ wildly in label rate (0.00 to 0.94,
printed above), so a single draw is high-variance.

Below I re-run the **exact same** Week-5 model and label three ways and print all three:

- **A) my actual W5 split** — single client-holdout draw, 20% of clients (seed 42)
- **B) the anti-pattern** — naive random-**row** split, where the same client leaks into
  both train and test (this is what inflates the score)
- **C) the honest estimate** — GroupKFold by `client_id` rotates **every** client through
  test once, giving a cross-client figure with no single-draw luck

The base rate (~0.54) is printed first and every metric is read against it.

In [1]:
# --- Section 2: my model under an honest split ---
# W5 already used a CLIENT-HOLDOUT split (good). But it reported a SINGLE draw of only
# ~6 held-out clients. This section shows three numbers on the SAME model + SAME label:
#   A) W5's actual single client-holdout draw (seed 42)      -> what I reported in W5
#   B) a naive random-ROW split                              -> the anti-pattern, inflates
#   C) GroupKFold rotating ALL clients through test          -> honest cross-client estimate
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42

def load_prepare(path="data/raw/content_refresh_anonymized.csv"):
    df = pd.read_csv(path)
    df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
    df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
    df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
    return df

def make_features(df):
    df = df.copy()
    df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
    df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
    df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
    df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
    numeric_feats = ["search_volume","competition","cpc","word_count","char_count",
        "log_impressions_90d","log_clicks_90d","log_sessions_90d","log_ai_sessions_90d",
        "days_with_impressions","days_with_sessions","content_age_days","days_since_last_update",
        "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
    cat_feats = ["competition_level","content_type","main_intent","age_tier",
        "freshness_tier","word_count_tier","impression_tier","position_tier"]
    for c in numeric_feats:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    for c in cat_feats:
        df[c] = df[c].fillna("unknown").astype(str)
    num = df[numeric_feats].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0)
    enc = pd.get_dummies(df[cat_feats].fillna("unknown").astype(str), prefix=cat_feats, dtype=float)
    X = pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)
    return X, df["is_declining_label"].astype(int), df["client_id"]

df = load_prepare()
X, y, clients = make_features(df)
print(f"Prepared {X.shape[0]:,} rows x {X.shape[1]} features across {clients.nunique()} clients")
print(f"Label rate (is_declining): {y.mean():.3f}   <- the base rate every metric must be read against")
client_label_rate = pd.Series(y.values, index=clients.values).groupby(level=0).mean()
print(f"Client-level label rate range: {client_label_rate.min():.2f} .. {client_label_rate.max():.2f}"
      "   <- clients differ a lot, so ONE held-out draw is high-variance")

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(y_true)[order[:k]].mean()

def metric_dict(y_true, scores):
    return {
        "precision@20": precision_at_k(y_true, scores, 20),
        "precision@50": precision_at_k(y_true, scores, 50),
        "precision@100": precision_at_k(y_true, scores, 100),
        "roc_auc": roc_auc_score(y_true, scores),
        "avg_precision": average_precision_score(y_true, scores),
    }

def fit_score(Xtr, ytr, Xte):
    rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
        min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE)
    rf.fit(Xtr, ytr)
    return rf.predict_proba(Xte)[:, 1]

# --- A) W5's ACTUAL split: hold out 20% of CLIENTS, single draw (reproduces w05_model.ipynb cell-6)
print("\n--- A) My W5 split: single client-holdout draw, 20% of clients (seed 42) ---")
all_idx = np.arange(len(X))
rng = np.random.default_rng(RANDOM_STATE)
uniq = df["client_id"].drop_duplicates().to_numpy()
shuffled = rng.permutation(uniq)
n_test = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test])
mask = df["client_id"].isin(test_clients).to_numpy()
tr_ix, te_ix = all_idx[~mask], all_idx[mask]
score_A = fit_score(X.iloc[tr_ix], y.iloc[tr_ix], X.iloc[te_ix])
A = metric_dict(y.iloc[te_ix], score_A)
print(f"  held-out clients={n_test}, test rows={len(te_ix):,}, test base rate={y.iloc[te_ix].mean():.3f}")
print(f"  P@20={A['precision@20']:.3f}  P@50={A['precision@50']:.3f}  P@100={A['precision@100']:.3f}"
      f"  ROC-AUC={A['roc_auc']:.3f}  AvgPr={A['avg_precision']:.3f}")

# --- B) The anti-pattern: naive random-ROW split (clients leak across train/test)
print("\n--- B) Anti-pattern: naive random-ROW split, 20% of rows (clients leak) ---")
idx = rng.permutation(len(X))
nr = int(round(len(X) * 0.2))
tr_b, te_b = idx[nr:], idx[:nr]
score_B = fit_score(X.iloc[tr_b], y.iloc[tr_b], X.iloc[te_b])
B = metric_dict(y.iloc[te_b], score_B)
print(f"  test rows={len(te_b):,}, test base rate={y.iloc[te_b].mean():.3f}")
print(f"  P@20={B['precision@20']:.3f}  P@50={B['precision@50']:.3f}  P@100={B['precision@100']:.3f}"
      f"  ROC-AUC={B['roc_auc']:.3f}  AvgPr={B['avg_precision']:.3f}")

# --- C) Honest cross-client estimate: GroupKFold rotates EVERY client through test once
print("\n--- C) Honest estimate: GroupKFold by client (every client tested once) ---")
gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(X))
for tr_i, te_i in gkf.split(X, y, clients):
    oof[te_i] = fit_score(X.iloc[tr_i], y.iloc[tr_i], X.iloc[te_i])
C = metric_dict(y, oof)
print(f"  every one of {clients.nunique()} clients held out once; base rate={y.mean():.3f}")
print(f"  P@20={C['precision@20']:.3f}  P@50={C['precision@50']:.3f}  P@100={C['precision@100']:.3f}"
      f"  ROC-AUC={C['roc_auc']:.3f}  AvgPr={C['avg_precision']:.3f}")

print("\n--- Before / after: same model, same label, three split designs ---")
print(f"  {'split':<40}{'base':>6}{'P@50':>8}{'ROC-AUC':>9}{'AvgPr':>8}")
for name, m, br in [
    ("A) W5 single client-holdout draw", A, y.iloc[te_ix].mean()),
    ("B) naive random-row (anti-pattern)", B, y.iloc[te_b].mean()),
    ("C) GroupKFold, all clients (honest)", C, y.mean())]:
    print(f"  {name:<40}{br:>6.2f}{m['precision@50']:>8.3f}{m['roc_auc']:>9.3f}{m['avg_precision']:>8.3f}")

print("\nInterpretation (honest reading):")
print(f"  - W5 already used a client-holdout split (the right KIND), but reported ONE draw:")
print(f"    P@50={A['precision@50']:.3f} on just {n_test} held-out clients (base rate {y.iloc[te_ix].mean():.2f}).")
print(f"  - Rotating ALL clients through test (GroupKFold) gives the honest cross-client")
print(f"    estimate P@50={C['precision@50']:.3f} (base rate {y.mean():.2f}). The single W5 draw was on the")
print(f"    optimistic side of real client-to-client variance.")
print(f"  - A naive random-ROW split would have inflated P@50 to {B['precision@50']:.3f} by letting the")
print(f"    same client sit in train and test -- exactly the mistake client-holdout avoids.")


Prepared 30,000 rows x 52 features across 32 clients
Label rate (is_declining): 0.542   <- the base rate every metric must be read against
Client-level label rate range: 0.00 .. 0.94   <- clients differ a lot, so ONE held-out draw is high-variance

--- A) My W5 split: single client-holdout draw, 20% of clients (seed 42) ---


  held-out clients=6, test rows=2,325, test base rate=0.391
  P@20=0.650  P@50=0.740  P@100=0.720  ROC-AUC=0.750  AvgPr=0.618

--- B) Anti-pattern: naive random-ROW split, 20% of rows (clients leak) ---


  test rows=6,000, test base rate=0.546
  P@20=1.000  P@50=0.980  P@100=0.930  ROC-AUC=0.764  AvgPr=0.781

--- C) Honest estimate: GroupKFold by client (every client tested once) ---


  every one of 32 clients held out once; base rate=0.542
  P@20=0.550  P@50=0.580  P@100=0.660  ROC-AUC=0.687  AvgPr=0.681

--- Before / after: same model, same label, three split designs ---
  split                                     base    P@50  ROC-AUC   AvgPr
  A) W5 single client-holdout draw          0.39   0.740    0.750   0.618
  B) naive random-row (anti-pattern)        0.55   0.980    0.764   0.781
  C) GroupKFold, all clients (honest)       0.54   0.580    0.687   0.681

Interpretation (honest reading):
  - W5 already used a client-holdout split (the right KIND), but reported ONE draw:
    P@50=0.740 on just 6 held-out clients (base rate 0.39).
  - Rotating ALL clients through test (GroupKFold) gives the honest cross-client
    estimate P@50=0.580 (base rate 0.54). The single W5 draw was on the
    optimistic side of real client-to-client variance.
  - A naive random-ROW split would have inflated P@50 to 0.980 by letting the
    same client sit in train and test -- exactly

## 3. Leakage audit

Same hunt as Week 3, now on my **final** feature set. The label
`is_declining = (trend_direction == "down")` is formed from a 30-day-vs-previous-30-day
impression change, so its window is the **last 30 days**. Any feature aggregated over a window
that overlaps those 30 days can leak the label.

The audit does three things:

1. classify every W5 numeric feature by its measurement window (point-in-time vs 30d/90d aggregate),
2. **deliberately add** the raw `trend_pct` (a direct label sibling) and watch the score jump —
   the leakage skill's "add a leaky feature, see it spike" check,
3. confirm none of the true label-sibling columns (`impressions_last_30d`, `trend_pct`, …) were
   ever passed as features.

In [2]:
# --- Section 3: leakage audit ---
# The label is:  is_declining = (trend_direction == "down"), and trend_direction comes
# from a <=30-day change vs the previous 30 days.  So ANY feature measured from a window
# that overlaps that same last-30-day (or crosses into it) can leak the label.

print("Step 1 — confirm the label's own formation window (the 'label provenance chain'):")
print("  is_declining  <-  trend_direction  <-  trend_pct = (imp_last_30d - imp_prev_30d)/imp_prev_30d")
print("  So the label window is the LAST 30 days.  Features must not peek at that window.\n")

# Track which of our W5 numeric features are aggregated over a window that touches/overlaps it.
feature_windows = [
    # per-day-ish or point-in-time columns: legal in principle, caveats below
    ("search_volume",       "point-in-time (kw-level)",       "legal"),
    ("competition",         "point-in-time (kw-level)",       "legal"),
    ("cpc",                 "point-in-time (kw-level)",       "legal"),
    ("word_count",          "static property",                "legal"),
    ("char_count",          "static property",                "legal"),
    ("content_age_days",    "point-in-time as of snapshot",   "legal"),
    ("days_since_last_update","point-in-time",                "legal"),
    ("ctr",                 "30d aggregate",                  "OVERLAPS label window -> leak risk"),
    ("avg_position",        "30d aggregate",                  "OVERLAPS label window -> leak risk"),
    ("engagement_rate",     "30d aggregate",                  "OVERLAPS label window -> leak risk"),
    ("scroll_rate",         "30d aggregate",                  "OVERLAPS label window -> leak risk"),
    ("ai_traffic_pct",      "30d aggregate",                  "OVERLAPS label window -> leak risk"),
    ("log_impressions_90d", "90d aggregate (OVRLPS last 30d)", "OVERLAPS label window -> leak risk"),
    ("log_clicks_90d",      "90d aggregate (OVRLPS last 30d)", "OVERLAPS label window -> leak risk"),
    ("log_sessions_90d",    "90d aggregate (OVRLPS last 30d)", "OVERLAPS label window -> leak risk"),
    ("log_ai_sessions_90d", "90d aggregate (OVRLPS last 30d)", "OVERLAPS label window -> leak risk"),
    ("days_with_impressions","90d aggregate",                  "OVERLAPS label window -> leak risk"),
    ("days_with_sessions",  "90d aggregate",                  "OVERLAPS label window -> leak risk"),
]
print("Step 2 — classify every W5 numeric feature by its measurement window:")
for name, kind, verdict in feature_windows:
    print(f"    {name:<24} {kind:<40} {verdict}")

leaky = [n for n,_,v in feature_windows if v.startswith("OVERLAPS")]
print(f"\n  -> {len(leaky)}/{len(feature_windows)} numeric features are 90-day or 30-day aggregates")
print(f"     that overlap the 30-day label window: {', '.join(leaky)}")
print("  These are legal to MEASURE but the window mismatch means the model can learn the trend")
print("  direction from the same window that produced it.  In W5 the label was also computed on")
print("  the SAME snapshot (same-window proxy), making it a 'same-artifact' comparison.\n")

# -------------------------------
# Step 3 — prove the leakage is real by ADDING the direct trend number as a feature.
# -------------------------------
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import average_precision_score, roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])
numeric_feats = ["search_volume","competition","cpc","word_count","char_count",
    "log_impressions_90d","log_clicks_90d","log_sessions_90d","log_ai_sessions_90d",
    "days_with_impressions","days_with_sessions","content_age_days","days_since_last_update",
    "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
cat_feats = ["competition_level","content_type","main_intent","age_tier","freshness_tier",
    "word_count_tier","impression_tier","position_tier"]

def build_X(df, extra=()):
    d = df.copy()
    for c in numeric_feats:
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0)
    for c in cat_feats:
        d[c] = d[c].fillna("unknown").astype(str)
    num = d[numeric_feats].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan).fillna(0)
    cat = pd.get_dummies(d[cat_feats].fillna("unknown").astype(str), prefix=cat_feats, dtype=float)
    out = pd.concat([num.reset_index(drop=True), cat.reset_index(drop=True)], axis=1)
    ext = pd.DataFrame(index=out.index)
    for name, series in extra:
        ext[name] = series.reset_index(drop=True)
    out = pd.concat([out, ext], axis=1)
    return out

# Panel A — every overlapping 90/30-day aggregate, with 5-fold random split.
Xa, ya = build_X(df), df["is_declining_label"].astype(int)
kf = KFold(5, shuffle=True, random_state=42)
oof_a = np.zeros(len(ya))
for tr, te in kf.split(Xa):
    rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
        min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=42)
    rf.fit(Xa.iloc[tr], ya.iloc[tr]);  oof_a[te] = rf.predict_proba(Xa.iloc[te])[:,1]
print(f"Panel A  (all W5 features, 5-fold random):  avg_pr={average_precision_score(ya, oof_a):.3f}  auc={roc_auc_score(ya, oof_a):.3f}")

# Panel B — same model but ADD the raw trend_pct as a feature.  This is the 'deliberately add a
# leaky feature and watch the score jump toward 1.0' check from the leakage skill.
XB = build_X(df, extra=[("trend_pct", df["trend_pct"])])
oof_b = np.zeros(len(ya))
for tr, te in kf.split(XB):
    rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10,
        min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=42)
    rf.fit(XB.iloc[tr], ya.iloc[tr]);  oof_b[te] = rf.predict_proba(XB.iloc[te])[:,1]
print(f"Panel B  (W5 features + raw trend_pct):      avg_pr={average_precision_score(ya, oof_b):.3f}  auc={roc_auc_score(ya, oof_b):.3f}")

print("\n  The score jump (Panel B vs Panel A) shows a direct label-sibling ('trend_pct') blows the")
print("  metric toward 1.0.  We deliberately did NOT feed that column; but the same-window overlap")
print("  is a milder form of the same leak and must be stated next to every W5 number.")

# Step 4 — confirm we never passed a TRUE label-sibling as a feature.
label_siblings = ["trend_direction","trend_pct","impressions_last_30d","impressions_prev_30d",
    "clicks_last_30d","clicks_prev_30d","sessions_last_30d","sessions_prev_30d"]
used = set(numeric_feats) | set(cat_feats)
present = [c for c in label_siblings if c in df.columns and c not in used]
print("Step 4 — label-sibling columns we never passed as features (good):")
print("   ", ", ".join(label_siblings))
print("   -> none of these appear in numeric_feats/cat_feats.  The W5 feature list is clean of")
print("      direct label siblings; its leakage is the window-overlap kind above.")


Step 1 — confirm the label's own formation window (the 'label provenance chain'):
  is_declining  <-  trend_direction  <-  trend_pct = (imp_last_30d - imp_prev_30d)/imp_prev_30d
  So the label window is the LAST 30 days.  Features must not peek at that window.

Step 2 — classify every W5 numeric feature by its measurement window:
    search_volume            point-in-time (kw-level)                 legal
    competition              point-in-time (kw-level)                 legal
    cpc                      point-in-time (kw-level)                 legal
    word_count               static property                          legal
    char_count               static property                          legal
    content_age_days         point-in-time as of snapshot             legal
    days_since_last_update   point-in-time                            legal
    ctr                      30d aggregate                            OVERLAPS label window -> leak risk
    avg_position             30

Panel A  (all W5 features, 5-fold random):  avg_pr=0.780  auc=0.765


Panel B  (W5 features + raw trend_pct):      avg_pr=1.000  auc=1.000

  The score jump (Panel B vs Panel A) shows a direct label-sibling ('trend_pct') blows the
  metric toward 1.0.  We deliberately did NOT feed that column; but the same-window overlap
  is a milder form of the same leak and must be stated next to every W5 number.
Step 4 — label-sibling columns we never passed as features (good):
    trend_direction, trend_pct, impressions_last_30d, impressions_prev_30d, clicks_last_30d, clicks_prev_30d, sessions_last_30d, sessions_prev_30d
   -> none of these appear in numeric_feats/cat_feats.  The W5 feature list is clean of
      direct label siblings; its leakage is the window-overlap kind above.


## 4. Claim rewrite

The boldest sentence I actually wrote in Week 5 was the first line of my summary cell:

**Before (my own words, verbatim from `w05_model.ipynb`):**
> "The random forest consistently beats both baselines across all metrics."

**Why this over-reaches — three problems the audit above exposes:**

1. **"consistently" rests on a single draw.** My W5 number (P@50 = 0.740, ROC-AUC = 0.750)
   came from *one* client-holdout draw of just **6 held-out clients** (test base rate 0.39).
   Section 2 rotates **all 32** clients through test with GroupKFold and P@50 falls to **0.580**
   and ROC-AUC to **0.687**. One draw is not "consistent"; the honest cross-client estimate is
   noticeably lower, so my single draw sat on the optimistic side of client-to-client variance.
2. **"across all metrics" hides the base rate.** P@50 = 0.580 has to be read against a **0.54**
   base rate — a real but modest lift, not a landslide. (A naive random-row split *looks* like a
   landslide at P@50 = 0.98, but only because the same client leaks into train and test.)
3. **the label is current-state, not a future outcome.** `is_declining` is a same-30-day-window
   trend label, and 11 of my 18 numeric features are 30/90-day aggregates that overlap that
   window (Section 3). So even the honest number describes "looks like it's declining *now*,"
   not "will decline" or "refreshing will recover it."

**After (what the evidence actually supports):**
> "Under an honest split that holds out whole clients (GroupKFold over all 32 clients), the
> random-forest ranking recovers currently-declining pages at **P@50 ≈ 0.58 against a 0.54
> base rate** — a directional lift over the Week-4 rule baseline (P@50 ≈ 0.38 on the same
> single-draw holdout where the forest scored 0.74). This is a **decision-support** signal for
> *which pages look like they are in decline today*, useful for prioritizing a review queue. It
> is **not** evidence that refreshing those pages will recover traffic, and the single-draw W5
> number (0.74) should be read as the optimistic end of a range whose honest center is ≈0.58."

Self-check against the paper's habits (Section 1) and the leakage rules (Section 3):
- [x] base rate (0.54) sits next to every P@50
- [x] reports the **range** (single-draw 0.74 → cross-client 0.58), not one flattering number
- [x] says "currently-declining / looks like," not "will decline" or "will recover"
- [x] names the window-overlap leak, not just the split
- [x] frames the output as ranking / decision support, not a causal refresh guarantee


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
